# Module 7.4 — Monitoring, Eval-in-Prod, and Drift Detection

**Goal:** Build the production monitoring loop for DeskMate — structured logging, online metrics, feedback capture, drift detection (KL + CUSUM), a retraining trigger, and a 4-panel dashboard. Determine the signal that tells you it's time to retrain.

## 1. Setup

In [ ]:
import os, re, json, time, uuid, random, math
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

SMOKE_TEST = True
random.seed(42)
np.random.seed(42)

os.makedirs('reports', exist_ok=True)
print('Config OK')

## 2. Simulate 500 Requests

In [ ]:
INTENTS     = ['billing_dispute', 'technical_bug', 'account_access', 'general_inquiry', 'refund']
ACTIONS     = ['reply', 'template_reply', 'escalate']

# Baseline intent distribution (matches training data)
BASELINE_DIST = {
    'billing_dispute': 0.35,
    'technical_bug':   0.28,
    'account_access':  0.18,
    'general_inquiry': 0.12,
    'refund':          0.07,
}

def simulate_request(ts_offset: int = 0, drift_phase: bool = False) -> dict:
    if drift_phase:
        # Simulate a product outage causing surge in technical_bug tickets
        dist = {'billing_dispute': 0.15, 'technical_bug': 0.55,
                'account_access': 0.12, 'general_inquiry': 0.10, 'refund': 0.08}
    else:
        dist = BASELINE_DIST
    intent     = random.choices(list(dist.keys()), weights=list(dist.values()))[0]
    confidence = np.clip(np.random.normal(0.82 if not drift_phase else 0.68, 0.10), 0.30, 1.00)
    faith      = np.clip(np.random.normal(0.74 if not drift_phase else 0.58, 0.12), 0.00, 1.00)
    action     = 'escalate' if confidence < 0.70 else random.choices(
        ['reply', 'template_reply'], [0.85, 0.15])[0]
    latency    = int(np.random.normal(1400 if not drift_phase else 1900, 300))
    return {
        'request_id':   str(uuid.uuid4())[:8],
        'ts_offset':    ts_offset,
        'intent':       intent,
        'confidence':   round(float(confidence), 4),
        'action':       action,
        'faithfulness': round(float(faith), 4),
        'latency_ms':   max(200, latency),
        'guardrail_flags': {
            'no_citation': faith < 0.40,
            'low_faith':   faith < 0.60,
            'short_reply': random.random() < 0.03,
        },
    }

# Generate 500 requests: first 350 baseline, then 150 drift
logs = [simulate_request(i, drift_phase=False) for i in range(350)]
logs += [simulate_request(350 + i, drift_phase=True) for i in range(150)]

print(f'Total log entries: {len(logs)}')
print(f'Sample: {logs[0]}')

## 3. Online Metrics — Rolling Window

In [ ]:
def compute_window_metrics(window: list[dict]) -> dict:
    if not window:
        return {}
    n = len(window)
    escalated = sum(1 for r in window if r['action'] == 'escalate')
    faiths    = [r['faithfulness'] for r in window]
    lats      = sorted(r['latency_ms'] for r in window)
    return {
        'n':                n,
        'escalation_rate':  round(escalated / n, 4),
        'mean_confidence':  round(sum(r['confidence'] for r in window) / n, 4),
        'faithfulness_p50': round(sorted(faiths)[n // 2], 4),
        'faithfulness_mean':round(sum(faiths) / n, 4),
        'no_citation_rate': round(sum(1 for r in window if r['guardrail_flags']['no_citation']) / n, 4),
        'latency_p50':      lats[n // 2],
        'latency_p95':      lats[int(n * 0.95)],
        'intent_dist':      {k: round(sum(1 for r in window if r['intent']==k)/n, 4)
                             for k in INTENTS},
    }

WINDOW_SIZE = 50
windows = [compute_window_metrics(logs[i:i+WINDOW_SIZE])
           for i in range(0, len(logs), WINDOW_SIZE)]

print(f'Windows computed: {len(windows)}')
print('Baseline window (W0):')
for k, v in windows[0].items():
    if k != 'intent_dist':
        print(f'  {k}: {v}')
print()
print('Drift window (W8, should show degradation):')
for k, v in windows[-1].items():
    if k != 'intent_dist':
        print(f'  {k}: {v}')

## 4. Feedback Capture

In [ ]:
FEEDBACK_DB: dict[str, dict] = {}

def record_feedback(request_id: str, thumbs_up: bool):
    FEEDBACK_DB[request_id] = {'thumbs_up': thumbs_up,
                               'ts': time.time()}

# Simulate explicit feedback: ~12% response rate, correlated with faithfulness
for r in logs:
    if random.random() < 0.12:  # 12% feedback rate
        # Higher faithfulness → more likely thumbs_up
        p_pos = r['faithfulness'] * 0.9 + 0.05
        record_feedback(r['request_id'], random.random() < p_pos)

n_feedback   = len(FEEDBACK_DB)
n_thumbs_up  = sum(1 for f in FEEDBACK_DB.values() if f['thumbs_up'])
thumbs_up_rate = round(n_thumbs_up / max(1, n_feedback), 3)

print(f'Requests with explicit feedback: {n_feedback} / {len(logs)} ({round(n_feedback/len(logs)*100,1)}%)')
print(f'Thumbs-up rate: {thumbs_up_rate}')

# Implicit feedback: reopened tickets
reopened = [r for r in logs if r['action'] == 'escalate']
print(f'Escalated (implicit negative): {len(reopened)} ({round(len(reopened)/len(logs)*100,1)}%)')

## 5. Intent Distribution Drift — KL Divergence

In [ ]:
def kl_divergence(p: dict, q: dict) -> float:
    keys  = list(INTENTS)
    p_arr = np.array([p.get(k, 1e-9) for k in keys], dtype=float)
    q_arr = np.array([q.get(k, 1e-9) for k in keys], dtype=float)
    p_arr /= p_arr.sum()
    q_arr /= q_arr.sum()
    return float(np.sum(p_arr * np.log(p_arr / q_arr)))

kl_over_time = [
    kl_divergence(BASELINE_DIST, w['intent_dist']) for w in windows
]

print('KL divergence per window:')
for i, (w, kl) in enumerate(zip(windows, kl_over_time)):
    drift_flag = 'DRIFT' if kl > 0.25 else '     ok'
    print(f'  W{i:<2}: KL={kl:.4f}  {drift_flag}')

## 6. CUSUM Detector — Faithfulness Drift

In [ ]:
def cusum(values: list[float], target: float, slack: float = 0.5,
          threshold: float = 5.0) -> tuple[list, int]:
    s, cusum_vals, first_alarm = 0.0, [], -1
    for i, v in enumerate(values):
        s = max(0.0, s + (target - v) - slack)
        cusum_vals.append(round(s, 4))
        if s > threshold and first_alarm == -1:
            first_alarm = i
    return cusum_vals, first_alarm

faith_means = [w['faithfulness_mean'] for w in windows]
baseline_faith = faith_means[0]

cusum_vals, alarm_idx = cusum(faith_means, target=baseline_faith)

print(f'Baseline faithfulness: {baseline_faith}')
print(f'CUSUM values: {cusum_vals}')
if alarm_idx >= 0:
    print(f'CUSUM alarm fired at window W{alarm_idx} '
          f'(faithfulness dropped to {faith_means[alarm_idx]:.3f})')
else:
    print('No CUSUM alarm fired.')

## 7. Retraining Trigger

In [ ]:
def should_retrain(metrics_24h: dict) -> tuple[bool, list]:
    reasons = []
    if metrics_24h.get('mean_confidence', 1.0) < 0.72:
        reasons.append('mean_confidence < 0.72')
    if metrics_24h.get('escalation_rate', 0.0) > 0.25:
        reasons.append('escalation_rate > 0.25')
    if metrics_24h.get('intent_kl', 0.0) > 0.25:
        reasons.append('intent_kl > 0.25')
    if metrics_24h.get('faithfulness_cusum', 0.0) > 5.0:
        reasons.append('faithfulness_cusum_alarm')
    return len(reasons) >= 2, reasons

# Evaluate per window
print('Retrain trigger evaluation per window:\n')
for i, (w, kl, cs) in enumerate(zip(windows, kl_over_time, cusum_vals)):
    metrics = {
        'mean_confidence':    w['mean_confidence'],
        'escalation_rate':    w['escalation_rate'],
        'intent_kl':          kl,
        'faithfulness_cusum': cs,
    }
    trigger, reasons = should_retrain(metrics)
    label = 'RETRAIN' if trigger else '      ok'
    print(f'  W{i:<2}: {label}  reasons={reasons}')

## 8. CI Regression Gate

In [ ]:
# In CI this runs against the real gold set (data/gold/).
# Here we simulate pass/fail with representative thresholds.

def run_ci_gate(candidate_metrics: dict, baseline_metrics: dict) -> dict:
    checks = {}
    checks['rouge_l'] = (
        candidate_metrics.get('rouge_l', 0) >=
        baseline_metrics.get('rouge_l', 0) - 0.01
    )
    checks['intent_f1'] = (
        candidate_metrics.get('intent_f1', 0) >=
        baseline_metrics.get('intent_f1', 0) - 0.01
    )
    checks['no_citation_rate'] = (
        candidate_metrics.get('no_citation_rate', 1.0) <= 0.05
    )
    checks['faithfulness_p50'] = (
        candidate_metrics.get('faithfulness_p50', 0) >= 0.70
    )
    passed = all(checks.values())
    return {'passed': passed, 'checks': checks}

BASELINE = {'rouge_l': 0.47, 'intent_f1': 0.88, 'no_citation_rate': 0.03, 'faithfulness_p50': 0.75}

# Candidate A: good retrain
cand_a = {'rouge_l': 0.49, 'intent_f1': 0.90, 'no_citation_rate': 0.02, 'faithfulness_p50': 0.78}
# Candidate B: bad retrain (regressed)
cand_b = {'rouge_l': 0.44, 'intent_f1': 0.86, 'no_citation_rate': 0.06, 'faithfulness_p50': 0.65}

result_a = run_ci_gate(cand_a, BASELINE)
result_b = run_ci_gate(cand_b, BASELINE)

print('Candidate A (good retrain):', 'PASS' if result_a['passed'] else 'FAIL')
for k, v in result_a['checks'].items():
    print(f'  {k}: {"pass" if v else "FAIL"}')
print()
print('Candidate B (regression):', 'PASS' if result_b['passed'] else 'FAIL')
for k, v in result_b['checks'].items():
    print(f'  {k}: {"pass" if v else "FAIL"}')

## 9. Dashboard — 4 Panels

In [ ]:
w_labels = [f'W{i}' for i in range(len(windows))]

fig = plt.figure(figsize=(14, 10))
gs  = gridspec.GridSpec(2, 2, hspace=0.4, wspace=0.35)

# Panel 1: Intent distribution — latest window vs baseline
ax1 = fig.add_subplot(gs[0, 0])
x   = np.arange(len(INTENTS))
w   = 0.35
latest_dist = windows[-1]['intent_dist']
ax1.bar(x - w/2, [BASELINE_DIST[k] for k in INTENTS],    w, label='Baseline', color='steelblue', alpha=0.8)
ax1.bar(x + w/2, [latest_dist.get(k,0) for k in INTENTS], w, label='Live',     color='coral',     alpha=0.8)
ax1.set_xticks(x)
ax1.set_xticklabels([i[:6] for i in INTENTS], rotation=20, fontsize=8)
ax1.set_title('Intent Distribution: Baseline vs Live')
ax1.set_ylabel('Fraction')
ax1.legend(fontsize=8)

# Panel 2: Quality metrics over time
ax2 = fig.add_subplot(gs[0, 1])
confidences  = [w['mean_confidence'] for w in windows]
faithfulness = [w['faithfulness_mean'] for w in windows]
esc_rates    = [w['escalation_rate'] for w in windows]
ax2.plot(w_labels, confidences,  'o-', color='steelblue', label='Mean confidence')
ax2.plot(w_labels, faithfulness, 's-', color='green',     label='Faithfulness mean')
ax2.plot(w_labels, esc_rates,    '^-', color='coral',     label='Escalation rate')
ax2.axhline(0.72, ls='--', color='steelblue', alpha=0.5, lw=0.8, label='Conf alert (0.72)')
ax2.axhline(0.25, ls='--', color='coral',     alpha=0.5, lw=0.8, label='Esc alert (0.25)')
ax2.set_title('Quality Metrics Over Time')
ax2.set_ylabel('Value')
ax2.set_xlabel('Window')
ax2.legend(fontsize=7)
ax2.tick_params(axis='x', rotation=30)

# Panel 3: Latency p50 / p95
ax3 = fig.add_subplot(gs[1, 0])
p50s = [w['latency_p50'] for w in windows]
p95s = [w['latency_p95'] for w in windows]
ax3.fill_between(range(len(windows)), p50s, p95s, alpha=0.2, color='purple', label='p50-p95 band')
ax3.plot(w_labels, p50s, 'o-', color='purple', label='p50')
ax3.plot(w_labels, p95s, 's-', color='indigo', label='p95')
ax3.axhline(4000, ls='--', color='red', alpha=0.5, lw=0.8, label='Alert (4000 ms)')
ax3.set_title('Latency (ms)')
ax3.set_ylabel('Latency (ms)')
ax3.set_xlabel('Window')
ax3.legend(fontsize=7)
ax3.tick_params(axis='x', rotation=30)

# Panel 4: Drift signals
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(w_labels, kl_over_time, 'o-', color='darkorange', label='KL divergence')
ax4.plot(w_labels, cusum_vals,   's-', color='crimson',    label='CUSUM (faithfulness)')
ax4.axhline(0.25, ls='--', color='darkorange', alpha=0.6, lw=0.8, label='KL alert')
ax4.axhline(5.0,  ls='--', color='crimson',    alpha=0.6, lw=0.8, label='CUSUM alarm')
if alarm_idx >= 0:
    ax4.axvline(alarm_idx, ls=':', color='crimson', alpha=0.7)
    ax4.annotate('CUSUM alarm', xy=(alarm_idx, cusum_vals[alarm_idx]),
                 xytext=(alarm_idx+0.3, cusum_vals[alarm_idx]+0.5),
                 fontsize=7, color='crimson')
ax4.set_title('Drift Signals')
ax4.set_ylabel('Signal value')
ax4.set_xlabel('Window')
ax4.legend(fontsize=7)
ax4.tick_params(axis='x', rotation=30)

plt.suptitle('DeskMate Production Monitoring Dashboard', fontsize=13, y=1.01)
plt.savefig('monitoring_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: monitoring_dashboard.png')

## 10. Checkpoint

In [ ]:
checkpoint = (
    'What signal tells you it is time to retrain?\n\n'
    'A retraining trigger fires when TWO OR MORE of these signals hold simultaneously\n'
    'for 24+ hours:\n'
    '  1. Mean encoder confidence < 0.72  (model uncertain about intent)\n'
    '  2. Escalation rate > 25%           (replies not resolving tickets)\n'
    '  3. Intent distribution KL > 0.25   (input distribution has shifted)\n'
    '  4. CUSUM alarm on faithfulness     (output quality steadily declining)\n\n'
    'Requiring two simultaneous signals reduces false positives from transient\n'
    'spikes (e.g. a product outage that floods support with one topic for a few\n'
    'hours — KL fires but confidence and faithfulness stay stable).\n\n'
    'The trigger dispatches a fine-tuning run on new data. The resulting candidate\n'
    'must pass an eval gate (ROUGE-L and F1 >= baseline - 0.01, no_citation_rate\n'
    '<= 5%, faithfulness_p50 >= 0.70) before promotion to production.'
)
print(checkpoint)

## 11. Save Report

In [ ]:
# Summary metrics
baseline_esc  = windows[0]['escalation_rate']
drift_esc     = windows[-1]['escalation_rate']
baseline_conf = windows[0]['mean_confidence']
drift_conf    = windows[-1]['mean_confidence']
max_kl        = round(max(kl_over_time), 4)
max_cusum     = round(max(cusum_vals), 4)

retrain_windows = []
for i, (w, kl, cs) in enumerate(zip(windows, kl_over_time, cusum_vals)):
    metrics = {'mean_confidence': w['mean_confidence'],
               'escalation_rate': w['escalation_rate'],
               'intent_kl': kl, 'faithfulness_cusum': cs}
    trigger, reasons = should_retrain(metrics)
    if trigger:
        retrain_windows.append(f'W{i} ({reasons})')

report = [
    '# Monitoring Report\n',
    f'Total requests simulated: {len(logs)}',
    f'Windows (size={WINDOW_SIZE}): {len(windows)}\n',
    '## Baseline vs Drift Window',
    '',
    f'Escalation rate:   baseline={baseline_esc}  drift={drift_esc}',
    f'Mean confidence:   baseline={baseline_conf}  drift={drift_conf}',
    f'Max KL divergence: {max_kl}',
    f'Max CUSUM value:   {max_cusum}',
    '',
    '## Retrain Trigger',
    '',
    f'Trigger fired in: {retrain_windows}',
    '',
    '## CI Gate',
    '',
    f'Candidate A (good): {"PASS" if result_a["passed"] else "FAIL"}',
    f'Candidate B (regression): {"PASS" if result_b["passed"] else "FAIL"}',
    '',
    '## Checkpoint',
    '',
    'Retrain trigger requires >= 2 signals: confidence<0.72, esc_rate>0.25,',
    'intent_kl>0.25, or faithfulness_cusum alarm.',
    'Two-signal rule prevents false positives from transient spikes.',
]

with open('reports/monitoring_report.md', 'w') as f:
    f.write('\n'.join(report))

print('Saved: reports/monitoring_report.md')
print('\n=== Module 7.4 Complete ===')